In [10]:
import mtl, pandas as pd, os 
#os.chdir("c:/Users/Gugu/Documents/UAntwerpen/mosis-assignment-1")
CURRENT_DIR = os.getcwd()

PROJECT_ROOT = os.path.abspath(
    os.path.join(CURRENT_DIR, "..", "..")
)

os.chdir(PROJECT_ROOT)

print(os.getcwd())

/Users/zlata/Library/CloudStorage/OneDrive-Personal/UAntwerpen/1_semester/Modelling_of_Software_Intensive_Systems/mosis_1


In [11]:
data=["data/stl/water heating/water_heating_system_kp50_ki50_kd10.csv",
     "data/stl/water heating/water_heating_system_kp100_ki0_kd10.csv",
      "data/stl/water heating/water_heating_system_kp100_ki1_kd40.csv",
       "data/stl/water heating/water_heating_system_kp100_ki5_kd10.csv",
        "data/stl/water heating/water_heating_system_kp100_ki20_kd10.csv",
         "data/stl/water heating/water_heating_system_kp200_ki2_kd40.csv",
          "data/stl/water heating/water_heating_system_kp300_ki0_kd45.csv",
           "data/stl/water heating/water_heating_system_kp300_ki10_kd20.csv",
             "data/stl/water heating/water_heating_system_kp500_ki0_kd50.csv",
             "data/stl/water heating/water_heating_system_kp500_ki5_kd0.csv"]

In [12]:
traces=[]

for file in data:
    frame = pd.read_csv(file)
    traces.append({
        "reaches_setpoint": list(zip(frame["time"], (frame["plant_block1.T_out"].abs()>=321.15)) ),
        "no_temperature_overshoot": list(zip(frame["time"], (frame["plant_block1.T_out"].abs()<=330)) ), 
        "settled_after_half_day": list(zip(frame["time"], (frame["plant_block1.T_out"].abs()>=320.15) & (frame["plant_block1.T_out"].abs()<=326.15)) ), 
        "level_in_band": list(zip(frame["time"], ((frame["plant_block1.level_out"].abs()>=1.4) & (frame["plant_block1.level_out"].abs()<=2.4))) ),
        "heater_stays_on": list(zip(frame["time"], (frame["pid_controller1.limiter.y"].abs() >=150.00)) ) 

    })

reaches_setpoint = mtl.parse("F[0, 86401] (reaches_setpoint)")
no_temperature_overshoot = mtl.parse("G[0, 86401] (no_temperature_overshoot)")
settled_after_half_day = mtl.parse("G[43200, 86401 ] (settled_after_half_day)")
level_in_band= mtl.parse("G[0, 86401] (level_in_band)")
heater_stays_on = mtl.parse("G[0, 86401] (heater_stays_on)")

properties = {
    "reaches_setpoint": reaches_setpoint,
    "no_temperature_overshoot": no_temperature_overshoot,
    "settled_after_half_day": settled_after_half_day,
    "level_in_band": level_in_band,
    "heater_stays_on": heater_stays_on

}


results = []

for i, trace in enumerate(traces):

    row = {
        "trace": os.path.basename(data[i])
    }

    for name, phi in properties.items():

        row[name] = phi( trace,time=0,quantitative=False,dt=1)

    results.append(row)
    

In [15]:
results_df = pd.DataFrame(results)

print(results_df)
results_df= results_df.replace({True: "true", False: "false"})

results_df.to_csv("results/stl/water_heating_results.csv", index=False)

                                      trace  reaches_setpoint  \
0   water_heating_system_kp50_ki50_kd10.csv              True   
1   water_heating_system_kp100_ki0_kd10.csv              True   
2   water_heating_system_kp100_ki1_kd40.csv              True   
3   water_heating_system_kp100_ki5_kd10.csv              True   
4  water_heating_system_kp100_ki20_kd10.csv              True   
5   water_heating_system_kp200_ki2_kd40.csv              True   
6   water_heating_system_kp300_ki0_kd45.csv              True   
7  water_heating_system_kp300_ki10_kd20.csv              True   
8   water_heating_system_kp500_ki0_kd50.csv              True   
9    water_heating_system_kp500_ki5_kd0.csv              True   

   no_temperature_overshoot  settled_after_half_day  level_in_band  \
0                     False                   False          False   
1                      True                   False          False   
2                     False                   False          False   
3   